# Travel Review Segmentation
**Assignment:** hierarchical clustering on Google attraction ratings.

No target. No train/test split. Cluster IDs 0/1/2/3 mean nothing until you read the means.

Class tools only: pandas, NumPy, matplotlib, scikit-learn, scipy.

Put `google_review_ratings.csv` next to this notebook. Run top to bottom.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

pd.set_option("display.max_columns", 30)
df = pd.read_csv("google_review_ratings.csv")
print(df.shape)
df.head()


## 1. Inspect
Three facts to write down before modeling: rows, missing columns, no target = clustering.


In [ ]:
print(df.info())
print(df.isna().sum())
print("Category 11 dtype:", df["Category 11"].dtype)
print("Unique users:", df["User"].nunique())
df.describe().T


PDF names for Category 1-24: churches, resorts, beaches, parks, theatres, museums, malls, zoo, restaurants, pubs/bars, local services, burger/pizza, hotels, juice bars, art galleries, dance clubs, pools, gyms, bakeries, beauty and spas, cafes, viewpoints, monuments, gardens.

Watch-outs:
- `User` is an ID. Drop it.
- `Unnamed: 25` is a junk column from a trailing comma. Drop it.
- `Category 11` is stored as text because one cell is garbled. Coerce before any median fill.


## 2. Clean
Median fill after numeric coerce. Do not median-fill a text column.


In [ ]:
df = df.drop(columns=["User", "Unnamed: 25"], errors="ignore")
df["Category 11"] = pd.to_numeric(df["Category 11"], errors="coerce")
print(df.isna().sum()[df.isna().sum() > 0])

for c in df.columns:
    df[c] = df[c].fillna(df[c].median())

print("NaNs left:", df.isna().sum().sum())
print(df.shape)

names = [
    "churches", "resorts", "beaches", "parks", "theatres", "museums",
    "malls", "zoo", "restaurants", "pubs_bars", "local_services", "burger_pizza",
    "hotels", "juice_bars", "art_galleries", "dance_clubs", "pools", "gyms",
    "bakeries", "beauty_spas", "cafes", "viewpoints", "monuments", "gardens",
]
df.columns = names
df.head()


## 3. Scale
Hierarchical clustering uses distance. Scale first or malls will dominate gyms.


In [ ]:
X = StandardScaler().fit_transform(df)
print(X.shape, "mean", round(float(X.mean()), 3), "std", round(float(X.std()), 3))


## 4. Dendrogram on a sample
Playbook rule: do not draw every leaf. Sample 800, Ward linkage, truncate. Cut where the vertical gap is tallest. That height is K.


In [ ]:
np.random.seed(42)
idx = np.random.choice(len(X), 800, replace=False)
Z = linkage(X[idx], method="ward")

plt.figure(figsize=(12, 5))
dendrogram(Z, truncate_mode="level", p=5)
plt.title("Ward dendrogram (800-user sample)")
plt.xlabel("users / merged groups")
plt.ylabel("distance")
plt.show()

print("Last 8 merge heights:")
print(np.round(Z[-8:, 2], 2))


## 5. Silhouette check (optional)
Low scores (~0.11-0.17) are normal on overlapping rating data. Still name groups from the means.


In [ ]:
print("sample Agglomerative Ward")
for k in range(2, 7):
    lab = AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(X[idx])
    print(k, "sil", round(silhouette_score(X[idx], lab), 3), "sizes", np.bincount(lab).tolist())

print("full file K-Means (compare only)")
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    lab = km.fit_predict(X)
    print(k, "inertia", round(km.inertia_, 1), "sil", round(silhouette_score(X, lab), 3))


## 6. Fit hierarchical labels
5,456 rows is fine for AgglomerativeClustering. Use the sample only for the picture.
k=3 or k=4 are the defensible cuts. Change one number if the instructor locks k.


In [ ]:
k = 4
labels = AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(X)
df["Cluster"] = labels
print(df["Cluster"].value_counts().sort_index())
print("silhouette:", round(silhouette_score(X, labels), 3))


## 7. Profile means and name the groups
Ignore the integers. Read restaurants / hotels / theatres / gardens.


In [ ]:
profile = df.groupby("Cluster").mean().round(2)
profile["n_users"] = df.groupby("Cluster").size()
print(profile.T)


Example talk track at k=4 (Ward, full file, after this cleaning):

| Cluster | Size | Story |
| --- | --- | --- |
| 0 | ~2585 | Culture / sightseeing -- parks, theatres, museums, beaches |
| 1 | ~1024 | Outdoor / quiet -- viewpoints, monuments, gardens, bakeries |
| 2 | ~1371 | Food + night -- malls, restaurants, pubs, zoo |
| 3 | ~476 | Lodging / convenience -- hotels, burger/pizza, juice bars |

If they lock k=3, Cluster 2 and 3 above merge into one city / hospitality group. Re-read the means before you name them.
